# Sarkari Ki Taiyari: nightly render
Started by GitHub Actions. Renders every script in `jobs/pending` of the public repo into a finished Reel.
GPU T4 x2, Internet On. Output: `/kaggle/working/out`.

In [ ]:
import os, subprocess, time
os.environ["MPLBACKEND"] = "Agg"   # InfiniteTalk crashes with the notebook plot backend
def run(cmd, tail=30):
    t=time.time(); p=subprocess.run(cmd,shell=True,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    if p.returncode: print("\n".join(p.stdout.splitlines()[-tail:])); raise RuntimeError("FAILED: "+cmd[:120])
    print(f"ok ({time.time()-t:.0f}s): {cmd[:100]}"); return p.stdout
REPO="https://github.com/mnandani09/sarkari-ki-taiyari-bot.git"
run(f"rm -rf /tmp/skt && git clone -q --depth 1 {REPO} /tmp/skt")
print(run("ls /tmp/skt/jobs/pending"))

In [ ]:
# fonts + browser for the cards and captions
os.makedirs(os.path.expanduser("~/.fonts"), exist_ok=True)
for w in ["Regular","Medium","SemiBold","Bold"]:
    run(f"wget -q -O ~/.fonts/Poppins-{w}.ttf https://raw.githubusercontent.com/google/fonts/main/ofl/poppins/Poppins-{w}.ttf")
run("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-noto-color-emoji", tail=10)
run("fc-cache -f")
run("pip install -q edge-tts==7.3.1 playwright soundfile")
run("python -m playwright install --with-deps chromium", tail=15)

In [ ]:
# InfiniteTalk (Wan2GP) in its own Python 3.11 environment
V="/tmp/wgpenv"; PY=f"{V}/bin/python"
run("pip install -q uv")
run(f"uv venv --python 3.11 --seed {V}")
run("git clone -q --depth 1 https://github.com/deepbeepmeep/Wan2GP.git /tmp/Wan2GP")
run(f"uv pip install --python {PY} torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128")
run(f"uv pip install --python {PY} -r /tmp/Wan2GP/requirements.txt", tail=40)

In [ ]:
# render every pending script (all Madam Ji clips in one InfiniteTalk session, checked automatically)
os.environ["WGP_PY"]=PY
p=subprocess.Popen("python -u /tmp/skt/render/pipeline.py /tmp/skt/jobs/pending /kaggle/working/out", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Warning" not in line and "ret = " not in line and "return _methods" not in line: print(line.rstrip()[:220], flush=True)
p.wait(); print("exit", p.returncode)
print(run("ls -la /kaggle/working/out"))